<a href="https://colab.research.google.com/github/Facrino/Analyse/blob/main/aviator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import math

# -------------------------------
# Fonctions
# -------------------------------

def duree_vol(mult):
    """Retourne la durée estimée d'un tour en secondes selon le multiplicateur"""
    if mult < 2:
        return 10
    elif 2 <= mult <= 10:
        return 40
    else:
        return 60

def poisson_exact(k, lam):
    """Probabilité d'avoir exactement k succès"""
    if lam < 0 or k < 0:
        return 0.0
    return (lam**k * math.exp(-lam)) / math.factorial(k)

def poisson_au_moins_1(lam):
    """Probabilité d'avoir au moins 1 succès"""
    if lam < 0:
        return 0.0
    return 1 - math.exp(-lam)

def proba_geometrique(k, p_theorique=0.10):
    """Loi Géométrique : Probabilité qu'un 10x tombe après k tours de vide"""
    return 1 - (1 - p_theorique)**k

def indice_pareto_alpha(data):
    """Indice de pression de Pareto (Chaleur du jeu)"""
    valeurs = [m for m in data if m >= 1.0]
    if len(valeurs) < 2: return 0.0
    somme_log = sum(math.log(m) for m in valeurs)
    return len(valeurs) / somme_log if somme_log > 0 else 0.0

# -------------------------------
# Entrée multiplicateurs
# -------------------------------
multiplicateurs_input = input(
    "Colle ici les multiplicateurs séparés par espace (ex: 4.0x 1.3x 2.79x ...):\n"
)

multiplicateurs = []
for m_str in multiplicateurs_input.split():
    cleaned_m_str = m_str.replace('x', '')
    try:
        multiplicateurs.append(float(cleaned_m_str))
    except ValueError:
        continue

# -------------------------------
# Calcul succès totaux et temps total
# -------------------------------
nb_succes_total = sum(1 for m in multiplicateurs if m >= 10)
temps_total_seconds = sum(duree_vol(m) for m in multiplicateurs)
temps_total_minutes = temps_total_seconds / 60

# -------------------------------
# Tableau tour par tour + Détections
# -------------------------------
cumul_seconds = 0
compteur_vide = 0
temps_35 = None
temps_65 = None

print(f"\n{'Tour':<4} | {'Mult.':<7} | {'Temps Cumulé':<12} | {'Poisson':<7} | {'Géom.':<7} | {'Pareto':<6}")
print("-" * 75)

for i, mult in enumerate(multiplicateurs, 1):
    duree = duree_vol(mult)
    cumul_seconds += duree
    t_minutes = cumul_seconds / 60

    # Gestion de la Loi Géométrique (compteur de tours sans 10x)
    if mult >= 10:
        compteur_vide = 0
    else:
        compteur_vide += 1

    # Calcul Poisson (selon ta méthode lambda proportionnel)
    if temps_total_minutes > 0:
        lam = nb_succes_total * (t_minutes / temps_total_minutes)
    else:
        lam = 0
    p_poisson = poisson_au_moins_1(lam)

    # Calcul Géométrie & Pareto
    p_geom = proba_geometrique(compteur_vide)
    alpha = indice_pareto_alpha(multiplicateurs[max(0, i-15):i])

    # Détection intervalle Poisson 35%-65%
    if temps_35 is None and p_poisson >= 0.35:
        temps_35 = t_minutes
    if temps_65 is None and p_poisson >= 0.65:
        temps_65 = t_minutes

    print(f"{i:4d} | {mult:5.2f}x | {int(t_minutes):02d}m {int(cumul_seconds%60):02d}s | {p_poisson*100:5.1f}% | {p_geom*100:5.1f}% | {alpha:.2f}")

# -------------------------------
# Résumé final (Tes statistiques)
# -------------------------------
print("\n" + "="*40)
print(f"Total tours : {len(multiplicateurs)} | Succès ≥10x : {nb_succes_total}")
print(f"Durée totale : {int(temps_total_minutes)} min {int(temps_total_seconds%60)} s")

# Probabilité Poisson pour le futur
temps_prochain_min = 15
if temps_total_minutes > 0:
    lam_prochain = nb_succes_total * (temps_prochain_min / temps_total_minutes)
    prob_2_succes = poisson_exact(2, lam_prochain)
    print(f"Probabilité 2 succès dans les {temps_prochain_min} prochaines min : {prob_2_succes*100:.2f}%")

# -------------------------------
# Synthèse des signaux (Poisson, Géom, Pareto)
# -------------------------------
print("-" * 40)
if temps_35 is not None and temps_65 is not None:
    print(f"INTERVALLE POISSON : {temps_35:.2f} min (35%) à {temps_65:.2f} min (65%)")

print(f"ÉTAT GÉOMÉTRIQUE   : {compteur_vide} tours de vide ({p_geom*100:.1f}%)")
status_pareto = "CHAUD 🔥" if alpha < 2.2 else "FROID ❄️" if alpha > 2.8 else "TIÈDE ⚖️"
print(f"PRESSION PARETO    : {alpha:.2f} -> {status_pareto}")

# Verdict
print("-" * 40)
if p_poisson >= 0.35 and p_geom >= 0.65:
    print("🟢 DÉCISION : MISEZ (Synchronisation Poisson/Géométrie)")
else:
    print("🔴 DÉCISION : ATTENDRE (Délai non atteint)")

Colle ici les multiplicateurs séparés par espace (ex: 4.0x 1.3x 2.79x ...):
3.41x 1.54x 1.10x 6.74x 1.48x 1.21x 4.53x 1.45x 8.21x 13.67x 1.68x 1.35x 1.37x 23.62x 1.26x 1.06x 2.60x 1.53x 17.24x 6.43x

Tour | Mult.   | Temps Cumulé | Poisson | Géom.   | Pareto
---------------------------------------------------------------------------
   1 |  3.41x | 00m 40s |  20.3% |  10.0% | 0.00
   2 |  1.54x | 00m 50s |  24.6% |  19.0% | 1.21
   3 |  1.10x | 01m 00s |  28.8% |  27.1% | 1.71
   4 |  6.74x | 01m 40s |  43.2% |  34.4% | 1.09
   5 |  1.48x | 01m 50s |  46.3% |  41.0% | 1.23
   6 |  1.21x | 02m 00s |  49.3% |  46.9% | 1.41
   7 |  4.53x | 02m 40s |  59.6% |  52.2% | 1.22
   8 |  1.45x | 02m 50s |  61.8% |  57.0% | 1.31
   9 |  8.21x | 03m 30s |  69.5% |  61.3% | 1.09
  10 | 13.67x | 04m 30s |  78.3% |   0.0% | 0.92
  11 |  1.68x | 04m 40s |  79.5% |  10.0% | 0.97
  12 |  1.35x | 04m 50s |  80.6% |  19.0% | 1.03
  13 |  1.37x | 05m 00s |  81.7% |  27.1% | 1.09
  14 | 23.62x | 06m 00s |  8

In [ ]:
import math

# -------------------------------
# Fonctions
# -------------------------------

def duree_vol(mult):
    if mult < 2:
        return 10
    elif 2 <= mult <= 10:
        return 40
    else:
        return 60

def poisson_exact(k, lam):
    if lam < 0 or k < 0:
        return 0.0
    return (lam**k * math.exp(-lam)) / math.factorial(k)

def poisson_au_moins_1(lam):
    if lam < 0:
        return 0.0
    return 1 - math.exp(-lam)

def proba_geometrique(k, p_theorique=0.10):
    return 1 - (1 - p_theorique)**k

def indice_pareto_alpha(data):
    valeurs = [m for m in data if m >= 1.0]
    if len(valeurs) < 2: return 0.0
    somme_log = sum(math.log(m) for m in valeurs)
    return len(valeurs) / somme_log if somme_log > 0 else 0.0

# -------------------------------
# Entrée multiplicateurs
# -------------------------------
multiplicateurs_input = input("Colle ici les multiplicateurs séparés par espace :\n")
multiplicateurs = []
for m_str in multiplicateurs_input.split():
    cleaned_m_str = m_str.replace('x', '')
    try:
        multiplicateurs.append(float(cleaned_m_str))
    except ValueError:
        continue

# -------------------------------
# Calculs Généraux
# -------------------------------
nb_succes_total = sum(1 for m in multiplicateurs if m >= 10)
temps_total_seconds = sum(duree_vol(m) for m in multiplicateurs)
temps_total_minutes = temps_total_seconds / 60
duree_moyenne_tour = temps_total_seconds / len(multiplicateurs) if len(multiplicateurs) > 0 else 0

# -------------------------------
# Tableau tour par tour
# -------------------------------
cumul_seconds = 0
compteur_vide = 0
temps_35 = None
temps_65 = None

print(f"\n{'Tour':<4} | {'Mult.':<7} | {'Temps Cumulé':<12} | {'Poisson':<7} | {'Géom.':<7}")
print("-" * 60)

for i, mult in enumerate(multiplicateurs, 1):
    duree = duree_vol(mult)
    cumul_seconds += duree
    t_minutes = cumul_seconds / 60

    if mult >= 10:
        compteur_vide = 0
    else:
        compteur_vide += 1

    if temps_total_minutes > 0:
        lam = nb_succes_total * (t_minutes / temps_total_minutes)
    else:
        lam = 0
    p_poisson = poisson_au_moins_1(lam)
    p_geom = proba_geometrique(compteur_vide)

    if temps_35 is None and p_poisson >= 0.35: temps_35 = t_minutes
    if temps_65 is None and p_poisson >= 0.65: temps_65 = t_minutes

    print(f"{i:4d} | {mult:5.2f}x | {int(t_minutes):02d}m {int(cumul_seconds%60):02d}s | {p_poisson*100:5.1f}% | {p_geom*100:5.1f}%")

# -------------------------------
# Synthèse Finale : MINUTES D'ATTENTE
# -------------------------------
print("\n" + "="*50)
print("🕒 SIGNAUX D'ATTENTE EN MINUTES")
print("="*50)

# --- SIGNAL GÉOMÉTRIQUE (MINUTES) ---
# Seuil 65% Géométrique = environ 10 tours de vide
tours_cible = 10
tours_restants = max(0, tours_cible - compteur_vide)
# Conversion des tours restants en minutes réelles
min_restantes_geom = (tours_restants * duree_moyenne_tour) / 60

print(f"LOI GÉOMÉTRIQUE :")
if min_restantes_geom > 0:
    print(f"⏳ Attendre encore : {min_restantes_geom:.2f} minutes ({tours_restants} tours)")
else:
    print(f"✅ SIGNAL GÉOMÉTRIQUE : MAINTENANT (Retard tours atteint)")

# --- SIGNAL POISSON (MINUTES) ---
print(f"\nLOI DE POISSON :")
if temps_65:
    # Calcul du délai par rapport au temps actuel
    delai_poisson = max(0, temps_65 - t_minutes)
    if delai_poisson > 0:
        print(f"⏳ Attendre encore : {delai_poisson:.2f} minutes")
    else:
        print(f"✅ SIGNAL POISSON : MAINTENANT (Pression session atteinte)")

# --- PARETO ---
alpha = indice_pareto_alpha(multiplicateurs[-15:])
status_pareto = "CHAUD 🔥" if alpha < 2.2 else "FROID ❄️" if alpha > 2.8 else "TIÈDE ⚖️"
print(f"\nTEMPÉRATURE DU JEU (Pareto) : {alpha:.2f} -> {status_pareto}")

print("="*50)

Colle ici les multiplicateurs séparés par espace :
3.41x 1.54x 1.10x 6.74x 1.48x 1.21x 4.53x 1.45x 8.21x 13.67x 1.68x 1.35x 1.37x 23.62x 1.26x 1.06x 2.60x 1.53x 17.24x 6.43x

Tour | Mult.   | Temps Cumulé | Poisson | Géom.  
------------------------------------------------------------
   1 |  3.41x | 00m 40s |  20.3% |  10.0%
   2 |  1.54x | 00m 50s |  24.6% |  19.0%
   3 |  1.10x | 01m 00s |  28.8% |  27.1%
   4 |  6.74x | 01m 40s |  43.2% |  34.4%
   5 |  1.48x | 01m 50s |  46.3% |  41.0%
   6 |  1.21x | 02m 00s |  49.3% |  46.9%
   7 |  4.53x | 02m 40s |  59.6% |  52.2%
   8 |  1.45x | 02m 50s |  61.8% |  57.0%
   9 |  8.21x | 03m 30s |  69.5% |  61.3%
  10 | 13.67x | 04m 30s |  78.3% |   0.0%
  11 |  1.68x | 04m 40s |  79.5% |  10.0%
  12 |  1.35x | 04m 50s |  80.6% |  19.0%
  13 |  1.37x | 05m 00s |  81.7% |  27.1%
  14 | 23.62x | 06m 00s |  87.0% |   0.0%
  15 |  1.26x | 06m 10s |  87.7% |  10.0%
  16 |  1.06x | 06m 20s |  88.4% |  19.0%
  17 |  2.60x | 07m 00s |  90.7% |  27.1%


In [ ]:
import math

# -------------------------------
# Fonctions
# -------------------------------

def duree_vol(mult):
    """Retourne la durée estimée d'un tour en secondes selon le multiplicateur"""
    if mult < 2:
        return 10
    elif 2 <= mult <= 10:
        return 40
    else:
        return 60

def poisson_exact(k, lam):
    if lam < 0 or k < 0:
        return 0.0
    return (lam**k * math.exp(-lam)) / math.factorial(k)

def poisson_au_moins_1(lam):
    if lam < 0:
        return 0.0
    return 1 - math.exp(-lam)

def indice_pareto_alpha(data):
    """Analyse de la puissance du jeu (Alpha Pareto)"""
    valeurs = [m for m in data if m >= 1.0]
    if len(valeurs) < 2: return 0.0
    somme_log = sum(math.log(m) for m in valeurs)
    return len(valeurs) / somme_log if somme_log > 0 else 0.0

# -------------------------------
# Entrée multiplicateurs
# -------------------------------
multiplicateurs_input = input(
    "Colle ici les multiplicateurs séparés par espace (ex: 4.0x 1.3x 2.79x ...):\n"
)

multiplicateurs = []
for m_str in multiplicateurs_input.split():
    cleaned_m_str = m_str.replace('x', '')
    try:
        multiplicateurs.append(float(cleaned_m_str))
    except ValueError:
        continue

# -------------------------------
# Calcul succès totaux et temps total
# -------------------------------
nb_succes_total = sum(1 for m in multiplicateurs if m >= 10)
temps_total_seconds = sum(duree_vol(m) for m in multiplicateurs)
temps_total_minutes = temps_total_seconds / 60
duree_moyenne_tour = temps_total_seconds / len(multiplicateurs) if len(multiplicateurs) > 0 else 0

# -------------------------------
# Tableau tour par tour
# -------------------------------
cumul_seconds = 0
compteur_vide = 0
temps_35 = None
temps_65 = None

print(f"\n{'Tour':<4} | {'Mult.':<7} | {'Temps Cumulé':<12} | {'Poisson %':<10} | {'Vide (Tours)':<12}")
print("-" * 65)

for i, mult in enumerate(multiplicateurs, 1):
    duree = duree_vol(mult)
    cumul_seconds += duree
    t_minutes = cumul_seconds / 60

    # Compteur Géométrique (tours depuis dernier 10x)
    if mult >= 10:
        compteur_vide = 0
    else:
        compteur_vide += 1

    # Ton calcul de Poisson original
    if temps_total_minutes > 0:
        lam = nb_succes_total * (t_minutes / temps_total_minutes)
    else:
        lam = 0
    p_poisson = poisson_au_moins_1(lam)

    if temps_35 is None and p_poisson >= 0.35: temps_35 = t_minutes
    if temps_65 is None and p_poisson >= 0.65: temps_65 = t_minutes

    print(f"{i:4d} | {mult:5.2f}x | {int(t_minutes):02d}m {int(cumul_seconds%60):02d}s | {p_poisson*100:8.1f}% | {compteur_vide:^12d}")

# -------------------------------
# Résumé final
# -------------------------------
print("\n" + "="*50)
print(f"RÉSUMÉ : {len(multiplicateurs)} tours | {nb_succes_total} succès (>=10x)")
print(f"DURÉE TOTALE : {int(temps_total_minutes)} min {int(temps_total_seconds%60)} s")

# -------------------------------
# 1. ANALYSE GÉOMÉTRIQUE (EN MINUTES)
# -------------------------------
# Seuil de probabilité 65% atteint environ après 10 tours sans succès
tours_cible_geom = 10
tours_restants = max(0, tours_cible_geom - compteur_vide)
minutes_restantes_geom = (tours_restants * duree_moyenne_tour) / 60

print(f"\n--- SIGNAL GÉOMÉTRIQUE ---")
if minutes_restantes_geom > 0:
    print(f"⏳ Attendre encore : {minutes_restantes_geom:.2f} MINUTES ({tours_restants} tours)")
else:
    print(f"✅ SIGNAL ATTEINT : Le retard de tours est suffisant.")

# -------------------------------
# 2. ANALYSE POISSON (TON INTERVALLE)
# -------------------------------
print(f"\n--- INTERVALLE POISSON (35%-65%) ---")
if temps_35 is not None and temps_65 is not None:
    print(f"Débuter à : {temps_35:.2f} min (35%)")
    print(f"Arrêter à : {temps_65:.2f} min (65%)")

    delai_avant_65 = max(0, temps_65 - t_minutes)
    if delai_avant_65 > 0:
        print(f"⏳ Temps restant avant 65% : {delai_avant_65:.2f} MINUTES")
else:
    print("Intervalle 35%-65% non atteint.")

# -------------------------------
# 3. ANALYSE PARETO (CHALEUR)
# -------------------------------
alpha = indice_pareto_alpha(multiplicateurs[-15:])
status_pareto = "CHAUD 🔥" if alpha < 2.2 else "FROID ❄️" if alpha > 2.8 else "TIÈDE ⚖️"
print(f"\n--- ÉTAT DU JEU (PARETO) ---")
print(f"Alpha : {alpha:.2f} -> {status_pareto}")

print("="*50)

Colle ici les multiplicateurs séparés par espace (ex: 4.0x 1.3x 2.79x ...):
3.41x 1.54x 1.10x 6.74x 1.48x 1.21x 4.53x 1.45x 8.21x 13.67x 1.68x 1.35x 1.37x 23.62x 1.26x 1.06x 2.60x 1.53x 17.24x 6.43x

Tour | Mult.   | Temps Cumulé | Poisson %  | Vide (Tours)
-----------------------------------------------------------------
   1 |  3.41x | 00m 40s |     20.3% |      1      
   2 |  1.54x | 00m 50s |     24.6% |      2      
   3 |  1.10x | 01m 00s |     28.8% |      3      
   4 |  6.74x | 01m 40s |     43.2% |      4      
   5 |  1.48x | 01m 50s |     46.3% |      5      
   6 |  1.21x | 02m 00s |     49.3% |      6      
   7 |  4.53x | 02m 40s |     59.6% |      7      
   8 |  1.45x | 02m 50s |     61.8% |      8      
   9 |  8.21x | 03m 30s |     69.5% |      9      
  10 | 13.67x | 04m 30s |     78.3% |      0      
  11 |  1.68x | 04m 40s |     79.5% |      1      
  12 |  1.35x | 04m 50s |     80.6% |      2      
  13 |  1.37x | 05m 00s |     81.7% |      3      
  14 | 23.62x

In [ ]:
import math

# -------------------------------
# Fonctions
# -------------------------------

def duree_vol(mult):
    """Retourne la durée estimée d'un tour en secondes"""
    if mult < 2:
        return 10
    elif 2 <= mult <= 10:
        return 40
    else:
        return 60

def poisson_exact(k, lam):
    if lam < 0 or k < 0:
        return 0.0
    return (lam**k * math.exp(-lam)) / math.factorial(k)

def poisson_au_moins_1(lam):
    if lam < 0:
        return 0.0
    return 1 - math.exp(-lam)

def indice_pareto_alpha(data):
    """Analyse de la puissance du jeu (Alpha Pareto)"""
    valeurs = [m for m in data if m >= 1.0]
    if len(valeurs) < 2: return 3.0 # Valeur par défaut (froid)
    somme_log = sum(math.log(m) for m in valeurs)
    return len(valeurs) / somme_log if somme_log > 0 else 3.0

# -------------------------------
# Entrée multiplicateurs
# -------------------------------
multiplicateurs_input = input(
    "Colle ici les multiplicateurs séparés par espace (ex: 4.0x 1.3x 2.79x ...):\n"
)

multiplicateurs = []
for m_str in multiplicateurs_input.split():
    cleaned_m_str = m_str.replace('x', '')
    try:
        multiplicateurs.append(float(cleaned_m_str))
    except ValueError:
        continue

# -------------------------------
# Calcul succès totaux et temps total
# -------------------------------
nb_succes_total = sum(1 for m in multiplicateurs if m >= 10)
temps_total_seconds = sum(duree_vol(m) for m in multiplicateurs)
temps_total_minutes = temps_total_seconds / 60
duree_moyenne_tour = temps_total_seconds / len(multiplicateurs) if len(multiplicateurs) > 0 else 0

# -------------------------------
# Tableau tour par tour
# -------------------------------
cumul_seconds = 0
compteur_vide = 0
temps_35 = None
temps_65 = None

print(f"\n{'Tour':<4} | {'Mult.':<7} | {'Temps Cumulé':<12} | {'Poisson %':<10} | {'Vide (T) ': <10}")
print("-" * 65)

for i, mult in enumerate(multiplicateurs, 1):
    duree = duree_vol(mult)
    cumul_seconds += duree
    t_minutes = cumul_seconds / 60

    if mult >= 10:
        compteur_vide = 0
    else:
        compteur_vide += 1

    if temps_total_minutes > 0:
        lam = nb_succes_total * (t_minutes / temps_total_minutes)
    else:
        lam = 0
    p_poisson = poisson_au_moins_1(lam)

    if temps_35 is None and p_poisson >= 0.35: temps_35 = t_minutes
    if temps_65 is None and p_poisson >= 0.65: temps_65 = t_minutes

    print(f"{i:4d} | {mult:5.2f}x | {int(t_minutes):02d}m {int(cumul_seconds%60):02d}s | {p_poisson*100:8.1f}% | {compteur_vide:^10d}")

# -------------------------------
# Résumé final
# -------------------------------
print("\n" + "="*50)
print(f"RÉSUMÉ : {len(multiplicateurs)} tours | {nb_succes_total} succès (>=10x)")

# -------------------------------
# 1. ANALYSE GÉOMÉTRIQUE (EN MINUTES)
# -------------------------------
tours_cible_geom = 10
tours_restants = max(0, tours_cible_geom - compteur_vide)
minutes_restantes_geom = (tours_restants * duree_moyenne_tour) / 60

print(f"\n--- SIGNAL GÉOMÉTRIQUE ---")
if minutes_restantes_geom > 0:
    print(f"⏳ Attendre encore : {minutes_restantes_geom:.2f} MINUTES ({tours_restants} tours)")
else:
    print(f"✅ SIGNAL ATTEINT (Retard suffisant)")

# -------------------------------
# 2. ANALYSE POISSON (INTERVALLE INTACT)
# -------------------------------
print(f"\n--- INTERVALLE POISSON (35%-65%) ---")
if temps_35 is not None and temps_65 is not None:
    print(f"Début zone (35%) : {temps_35:.2f} min")
    print(f"Fin zone   (65%) : {temps_65:.2f} min")

    delai_65 = max(0, temps_65 - t_minutes)
    if delai_65 > 0:
        print(f"⏳ Temps restant avant 65% : {delai_65:.2f} MINUTES")
else:
    print("Intervalle 35%-65% non atteint.")

# -------------------------------
# 3. CÔTE À VISER (BASÉE SUR PARETO)
# -------------------------------
alpha = indice_pareto_alpha(multiplicateurs[-15:])

# Calcul de la côte recommandée selon la distribution de Pareto
# Formule simplifiée : Côte = xmin * (1/U)^(1/alpha) où U est la probabilité cible
# Plus Alpha est bas, plus la côte recommandée augmente.
if alpha < 2.0:
    cote_visee = "15x - 30x (HAUTE 🔥)"
elif alpha < 2.5:
    cote_visee = "10x - 15x (MOYENNE ⚖️)"
else:
    cote_visee = "2x - 5x (BASSE ❄️)"

print(f"\n--- ANALYSE DE LA PUISSANCE ---")
print(f"Indice Alpha : {alpha:.2f}")
print(f"🎯 CÔTE À VISER : {cote_visee}")

print("="*50)

Colle ici les multiplicateurs séparés par espace (ex: 4.0x 1.3x 2.79x ...):
3.41x 1.54x 1.10x 6.74x 1.48x 1.21x 4.53x 1.45x 8.21x 13.67x 1.68x 1.35x 1.37x 23.62x 1.26x 1.06x 2.60x 1.53x 17.24x 6.43x

Tour | Mult.   | Temps Cumulé | Poisson %  | Vide (T)  
-----------------------------------------------------------------
   1 |  3.41x | 00m 40s |     20.3% |     1     
   2 |  1.54x | 00m 50s |     24.6% |     2     
   3 |  1.10x | 01m 00s |     28.8% |     3     
   4 |  6.74x | 01m 40s |     43.2% |     4     
   5 |  1.48x | 01m 50s |     46.3% |     5     
   6 |  1.21x | 02m 00s |     49.3% |     6     
   7 |  4.53x | 02m 40s |     59.6% |     7     
   8 |  1.45x | 02m 50s |     61.8% |     8     
   9 |  8.21x | 03m 30s |     69.5% |     9     
  10 | 13.67x | 04m 30s |     78.3% |     0     
  11 |  1.68x | 04m 40s |     79.5% |     1     
  12 |  1.35x | 04m 50s |     80.6% |     2     
  13 |  1.37x | 05m 00s |     81.7% |     3     
  14 | 23.62x | 06m 00s |     87.0% |    